In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import sim_ranking as sr
import ml_tools as mlt

%matplotlib inline

In [ ]:
def display_scenario(event_id: str, site_int: str, gnn_results_df: pd.DataFrame, dist_matrix: pd.DataFrame, obs_data: sr.ObservedData, 
                     cim_results_df: pd.DataFrame = None, n_obs_sites: int = 5, emp_gmm_params: pd.DataFrame = None):    
    display(Markdown(f"### {site_int}"))
    if f"{event_id}_{site_int}" in gnn_results_df.index:
        # Plots
        cur_obs_sites = sr.plot_ind_scenarios.get_obs_sites(event_id, site_int, gnn_results_df, dist_matrix, n_obs_sites=n_obs_sites)
        fig = sr.plot_ind_scenarios.create_2plot_log(event_id, site_int, cur_obs_sites, gnn_results_df, cim_results_df, obs_data, dist_matrix, emp_gmm_params=emp_gmm_params)
        plt.show(fig)
        plt.close(fig)

        # Info
        site_info_df = sr.plot_ind_scenarios.get_site_info_df(event_id, site_int, cur_obs_sites, obs_data, dist_matrix)
        display(site_info_df)
    else:
        print(f"No results for {event_id}_{site_int}")

In [ ]:
wdata = Path("/Users/claudy/dev/work/data")
model_dir = Path("/Users/claudy/dev/work/data/sim_ranking/results/gnn/0218_2000_full_v4p2NZGMDB_v2p1GNN")
rel_pred_path = "3468575/gnn_predictions_withoutSelf.parquet"
cim_results_ffp = Path("/Users/claudy/dev/work/data/sim_ranking/results/cIM/0219_3468575/cIM_results.parquet")

In [ ]:
event_id = "3468575"
run_config = sr.ml.RunConfig.from_yaml(model_dir / "run_config.yaml")

In [ ]:
# Load NZGMDB data
obs_data = sr.data.load_obs_nzgmdb(wdata / run_config.rel_obs_data_ffp)
obs_sites = obs_data.event_sites[event_id]

In [ ]:
# Distance matrix
dist_matrix = sr.utils.calculate_distance_matrix(obs_data.sites, obs_data.site_df)

In [ ]:
# Load the empirical GMM predictions
marg_pred_df = sr.data.load_emp_gmm_params( wdata / "sim_ranking/emp_gm_params/nzgmdb_v4p2" / "emp_gm_params_400MaxRjb.parquet" )
marg_pred_df = marg_pred_df[marg_pred_df["event_id"] == event_id]
marg_pred_df[sr.constants.PSA_KEYS] = np.log(obs_data.record_df.loc[marg_pred_df.index, sr.constants.PSA_KEYS])
marg_pred_df["site_int"] = marg_pred_df["site_id"]

In [ ]:
# Load the cIM results
cim_results_df = pd.read_parquet(cim_results_ffp)
assert cim_results_df.iloc[0].event_id == event_id
cim_results_df = cim_results_df.loc[cim_results_df["site_int"].isin(obs_sites)]

In [ ]:
# Load GNN results
gnn_pred_df = pd.read_parquet(model_dir / rel_pred_path)
gnn_pred_df = gnn_pred_df.loc[np.isin(gnn_pred_df["site_int"], obs_sites)]

# Add observed GM at the SOIs (if available)
gnn_pred_df[run_config.ims] = np.log(obs_data.record_df.loc[gnn_pred_df.index, run_config.ims])

### Bias And Residual Standard Deviation

In [ ]:
# Compute residuals, bias, and residual standard deviation
gnn_res_df = sr.ml.gnn_gm.get_residuals(gnn_pred_df, run_config.ims)
gnn_bias = gnn_res_df[run_config.ims].mean(axis=0)
gnn_res_std = gnn_res_df[run_config.ims].std(axis=0)

cim_res_df = sr.ml.gnn_gm.get_residuals(cim_results_df, sr.constants.PSA_KEYS, pred_suffix="cond_mean")
cim_bias = cim_res_df[sr.constants.PSA_KEYS].mean(axis=0)
cim_res_std = cim_res_df[sr.constants.PSA_KEYS].std(axis=0)

marg_res_df = sr.ml.gnn_gm.get_residuals(marg_pred_df, sr.constants.PSA_KEYS, pred_suffix="mean")
marg_bias = marg_res_df[sr.constants.PSA_KEYS].mean(axis=0)
marg_res_std = marg_res_df[sr.constants.PSA_KEYS].std(axis=0)

In [ ]:
# Bias & Residual Standard Deviation Plot
fig, ax1, ax2, ax3, ax4 =  sr.plot_utils.get_bias_residual_fig(bias_y_axis_limits=(-1.5, 1.5))

ax1.semilogx(sr.constants.PERIODS, gnn_bias[sr.constants.PSA_KEYS], label="GNN", c="darkblue")
ax1.semilogx(sr.constants.PERIODS, cim_bias[sr.constants.PSA_KEYS], label="cIM", c="r")
ax1.semilogx(sr.constants.PERIODS, marg_bias[sr.constants.PSA_KEYS], label="Empirical GMM", c="gray")

ax2.scatter(sr.constants.NON_PSA_IMs, gnn_bias[sr.constants.NON_PSA_IMs], c="darkblue")
ax2.xaxis.set_tick_params(rotation=90)

ax3.semilogx(sr.constants.PERIODS, gnn_res_std[sr.constants.PSA_KEYS], label="GNN", c="darkblue")
ax3.semilogx(sr.constants.PERIODS, cim_res_std[sr.constants.PSA_KEYS], label="cIM", c="r")
ax3.semilogx(sr.constants.PERIODS, marg_res_std[sr.constants.PSA_KEYS], label="Empirical GMM", c="gray")

ax4.scatter(sr.constants.NON_PSA_IMs, gnn_res_std[sr.constants.NON_PSA_IMs], c="darkblue")
ax4.xaxis.set_tick_params(rotation=90)

ax1.legend()

### Individual Site Predictions

In [ ]:
for cur_site_int in gnn_pred_df.site_int.values:
    display_scenario(event_id, cur_site_int, gnn_pred_df, dist_matrix, obs_data, n_obs_sites=run_config.max_n_obs_sites, emp_gmm_params=marg_pred_df, cim_results_df=cim_results_df)